In [1]:
import pandas as pd
import requests

import geopandas as gpd
import pandas as pd
from rasterstats import zonal_stats
import glob
import os

In [2]:
import re

def get_key_from_environment(file_path: str, key: str) -> str | None:
    """
    Extracts a key from Angular's environment.ts file.

    Args:
        file_path: Path to environment.ts
        key: The key name (e.g., "apiKey")

    Returns:
        The value as a string, or None if not found.
    """
    with open(file_path, "r", encoding="utf-8") as f:
        content = f.read()

    # Regex to match key: 'value' or key: "value"
    pattern = rf'{key}\s*:\s*[\'"]([^\'"]+)[\'"]'
    match = re.search(pattern, content)

    return match.group(1) if match else None


# Example usage
file_path = "../src/environments/environment.ts"
api_key = get_key_from_environment(file_path, "apiToken")



In [3]:
header = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json"
}

from datetime import datetime
from dateutil.relativedelta import relativedelta

scale = 12
start_date = datetime(2015, 1, 1)   # Sept 2024
end_date = datetime(2025, 8, 1)     # Aug 2025

# Loop over months
date = start_date
while date <= end_date:
    date_str = date.strftime("%Y-%m")
    url = f"https://api.hcdp.ikewai.org/raster?datatype=rainfall&production=new&period=month&date={date_str}"
    
    print(f"Fetching {url} ...")
    res = requests.get(url, headers=header)
    
    if res.status_code == 200:
        file = f"./data/rainfall_{date_str}.tif"
        with open(file, "wb") as f:
            f.write(res.content)
    
    # Move to next month
    date += relativedelta(months=1)

Fetching https://api.hcdp.ikewai.org/raster?datatype=rainfall&production=new&period=month&date=2015-01 ...
Fetching https://api.hcdp.ikewai.org/raster?datatype=rainfall&production=new&period=month&date=2015-02 ...
Fetching https://api.hcdp.ikewai.org/raster?datatype=rainfall&production=new&period=month&date=2015-03 ...
Fetching https://api.hcdp.ikewai.org/raster?datatype=rainfall&production=new&period=month&date=2015-04 ...
Fetching https://api.hcdp.ikewai.org/raster?datatype=rainfall&production=new&period=month&date=2015-05 ...
Fetching https://api.hcdp.ikewai.org/raster?datatype=rainfall&production=new&period=month&date=2015-06 ...
Fetching https://api.hcdp.ikewai.org/raster?datatype=rainfall&production=new&period=month&date=2015-07 ...
Fetching https://api.hcdp.ikewai.org/raster?datatype=rainfall&production=new&period=month&date=2015-08 ...
Fetching https://api.hcdp.ikewai.org/raster?datatype=rainfall&production=new&period=month&date=2015-09 ...
Fetching https://api.hcdp.ikewai.org/

In [11]:
#statewide
shapefile = "../public/Coastline.shp"
raster_folder = "./data"

# Load shapefile and dissolve to one feature (statewide)
gdf = gpd.read_file(shapefile)
gdf_statewide = gdf.dissolve()  # merges all polygons into one

records = []

for tif in sorted(glob.glob(os.path.join(raster_folder, f"rainfall_*.tif"))):
    # Extract date from filename (spi001_2024-09.tif → 2024-09)
    date = os.path.basename(tif).split("_")[1].replace(".tif", "")
    
    # Compute zonal stats for the dissolved geometry
    stats = zonal_stats(
        vectors=gdf_statewide,
        raster=tif,
        stats=["mean"],
        geojson_out=False,
        nodata=-9999
    )
    
    records.append({
        "date": date,
        "value": stats[0]["mean"]
    })


# Pivot to wide format: one row (statewide), months as columns
df = pd.DataFrame(records).set_index("date").T
df.insert(0, "state", "statewide")  # add proper state column
df.to_csv(f"../public/statewide_rf{scale}.csv", index=False)

print(df)

date       state    2015-01    2015-02     2015-03     2015-04     2015-05  \
value  statewide  89.359712  80.079705  146.086678  157.398291  136.887259   

date      2015-06     2015-07     2015-08     2015-09  ...       2024-11  \
value  101.936873  110.644671  301.622934  327.227254  ... -1.209590e+33   

date        2024-12       2025-01       2025-02       2025-03       2025-04  \
value -1.209590e+33 -1.209590e+33 -1.209590e+33 -1.209590e+33 -1.209590e+33   

date        2025-05       2025-06       2025-07       2025-08  
value -1.209590e+33 -1.209590e+33 -1.209590e+33 -1.209590e+33  

[1 rows x 129 columns]


In [12]:

folder = "./data"

for filename in os.listdir(folder):
    file_path = os.path.join(folder, filename)
    if os.path.isfile(file_path):  # ensures it's a file
        os.remove(file_path)